### Import needed libraries

In [104]:
import os
import pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical # type: ignore
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import LSTM, Dense, Masking, Dropout # type: ignore
from tensorflow.keras.layers import (LSTM, Dense, Masking, Dropout, BatchNormalization, 
                                   Bidirectional, Conv1D, MaxPooling1D, GlobalMaxPooling1D,
                                   Input, Concatenate, Attention, MultiHeadAttention,
                                   LayerNormalization, TimeDistributed)
from tensorflow.keras.regularizers import l1_l2
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau, ModelCheckpoint,
                                      TensorBoard, LearningRateScheduler)

In [105]:
X = []
y = []

data_dir = "pickles/"

for file in os.listdir(data_dir):
    file_path = os.path.join(data_dir, file)
    if os.path.exists(file_path):
        with open(file_path, "rb") as f:
            data = pickle.load(f)
        for entry in data:
            points = entry["points"]  # list of frames, each frame = list of (x,y)
            if points:
                # Flatten each frame into 1D vector
                seq = [np.array(frame, dtype=np.float32).flatten() for frame in points]
                X.append(seq)
                y.append(entry["class_name"])
    else:
        print(f"{file} not found!")

print(f"Loaded {len(X)} sequences")

Loaded 92 sequences


In [106]:
max_seq_len = max(len(seq) for seq in X)
feature_dim = max(len(frame) for seq in X for frame in seq)  # largest frame vector size

X_padded = []
for seq in X:
    arr = np.zeros((max_seq_len, feature_dim), dtype=np.float32)
    for i, frame in enumerate(seq):
        arr[i, :len(frame)] = frame
    X_padded.append(arr)

X_padded = np.array(X_padded, dtype=np.float32)  # (num_samples, max_seq_len, feature_dim)
print("X_padded shape:", X_padded.shape)
print("X_padded:", X_padded[43][0])

le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_onehot = to_categorical(y_encoded)

X_padded shape: (92, 20, 426)
X_padded: [228.21819 241.5293  230.6384  242.7394  230.6384  242.7394  233.0586
 243.9495  233.0586  243.9495  237.89899 246.3697  237.89899 246.3697
 242.7394  248.7899  242.7394  248.7899  250.      250.      250.
 250.      257.2606  250.      257.2606  250.      265.7313  250.
 265.7313  250.      271.7818  248.7899  271.7818  248.7899  276.62222
 248.7899  276.62222 248.7899  280.2525  247.5798  228.21819 241.5293
 229.42828 239.1091  229.42828 239.1091  233.0586  235.47879 233.0586
 235.47879 236.68889 231.8485  236.68889 231.8485  243.9495  228.21819
 243.9495  228.21819 251.2101  229.42828 251.2101  229.42828 259.68082
 229.42828 259.68082 229.42828 268.15152 235.47879 268.15152 235.47879
 274.20203 240.3192  274.20203 240.3192  279.04242 243.9495  279.04242
 243.9495  280.2525  247.5798  233.0586  241.5293  235.47879 240.3192
 235.47879 240.3192  237.89899 240.3192  237.89899 240.3192  241.5293
 239.1091  241.5293  239.1091  245.15959 239.1091  24

In [111]:
X_train, X_test, y_train, y_test = train_test_split(
    X_padded, y_onehot, test_size=0.2, random_state=42, stratify=y_encoded
)

X_train, X_test, y_train, y_test = train_test_split(
    X_padded, y_onehot, test_size=0.2, random_state=42, stratify=y_encoded
)

model = Sequential([
    Masking(mask_value=0.0, input_shape=(X_padded.shape[1], X_padded.shape[2])),
    LSTM(128, return_sequences=True, recurrent_dropout=0.2),
    BatchNormalization(),
    LSTM(64, return_sequences=False, recurrent_dropout=0.2),
    Dropout(0.3),
    Dense(y_onehot.shape[1], activation="softmax")
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True) # type: ignore
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5) # type: ignore

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=16,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, reduce_lr]
)

loss, acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {acc:.3f}")


/home/yassin/miniconda3/envs/tf/lib/python3.13/site-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_25"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking_27 (Masking)            │ (None, 20, 426)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_44 (LSTM)                  │ (None, 20, 128)        │       284,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_20          │ (None, 20, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_45 (LSTM)                  │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_31 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 334,340 (1.28 MB)

 Trainable params: 334,084 (1.27 MB)

 Non-trainable params: 256 (1.00 KB)

Epoch 1/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 217ms/step - accuracy: 0.2603 - loss: 1.4510 - val_accuracy: 0.3158 - val_loss: 1.4663 - learning_rate: 0.0010
Epoch 2/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.6164 - loss: 1.1171 - val_accuracy: 0.3158 - val_loss: 1.3965 - learning_rate: 0.0010
Epoch 3/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.8082 - loss: 0.9409 - val_accuracy: 0.3158 - val_loss: 1.2836 - learning_rate: 0.0010
Epoch 4/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.8219 - loss: 0.7691 - val_accuracy: 0.3158 - val_loss: 1.2427 - learning_rate: 0.0010
Epoch 5/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.8630 - loss: 0.6959 - val_accuracy: 0.3158 - val_loss: 1.2044 - learning_rate: 0.0010
Epoch 6/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.8630 - loss: 0.5593 - val_accuracy: 0.3158 - val_loss: 1.1781 - learning_rate: 0.0010
Epoch 7/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.8630 - loss: 0.4648 - val_accuracy: 0.3158

In [108]:
model.save("gesture_model.h5", save_format="h5")

In [109]:
# import libraries
with open('label_encoder.pkl', 'wb') as f:
  pickle.dump(le, f)